# 과제 2. LLM 기반 복원 모델 구현 (전이학습)
### 모델: beomi/gemma-ko-2b

## 1. 라이브러리 임포트

In [1]:
!pip install trl

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import TrainingArguments
from trl import SFTTrainer
from sklearn.model_selection import train_test_split


## 2. Todo: 데이터 로드

In [3]:
from google.colab import drive
drive.mount('/content/drive')

train = pd.read_csv('/content/drive/MyDrive/hangeul-restoration-teamproject/train.csv')
test  = pd.read_csv('/content/drive/MyDrive/hangeul-restoration-teamproject/test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print(train.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train shape: (11263, 3)
Test shape: (1689, 2)
            ID                                              input  \
0  TRAIN_00000  별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈...   
1  TRAIN_00001                             잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ   
2  TRAIN_00002                                    절테 간면 않 된는 굣 멥몫   
3  TRAIN_00003  야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵...   
4  TRAIN_00004  집윈 축쳐눌료 딴너왓눈뎁 카셩뷔 좋곱 칼쿰한네올. 쩌럼한뒈 뮬콰 욺료토 잊쿄 빻토 ...   

                                              output  
0  별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자...  
1                             잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ  
2                                    절대 가면 안 되는 곳 메모  
3  아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵...  
4  지인 추천으로 다녀왔는데 가성비 좋고 깔끔하네요. 저렴한데 물과 음료도 있고 방도 ...  


## 3. TODO: Train / Validation 데이터 분리


In [4]:
from sklearn.model_selection import train_test_split

# TODO: 아래 코드를 완성하세요. (힌트: test_size는 0.1~0.2, random_state=42 고정)
train_data, val_data =  train_test_split(train, test_size = 0.1, random_state=42)


print('train_data:', len(train_data))
print('val_data:', len(val_data))


train_data: 10136
val_data: 1127


## 4. 모델 로드 (4bit 양자화)

In [5]:
!pip install -U transformers accelerate bitsandbytes peft trl

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = 'beomi/gemma-ko-2b'
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

In [7]:
def create_prompt(input_text, output_text=''):
    # TODO: 아래 messages 안의 "system" content 내용을 직접 설계해보세요.
    # 모델에게 줄 역할, 제약 조건, Few-shot(예시)을 자유롭게 작성하여 성능을 끌어올려 보세요!

    system_prompt = (
        "당신은 난독화된 한국어 리뷰를 자연스럽고 정확한 문장으로 복원하는 전문가이다.\n"
        "반드시 복원된 문장만 출력하라.\n\n"

        "Example 1:\n"
        "Input: 배쏭 개빠름 진짜 죠아용\n"
        "Output: 배송 개빠름 진짜 좋아용\n\n"

        "Example 2:\n"
        "Input: 마싯구 양 많아여!!\n"
        "Output: 맛있고 양 많아요!!\n\n"
    )

    prompt = (
        f"{system_prompt}"
        f"Input: {input_text}\n"
        f"Output: "
    )

    if output_text:
        prompt += output_text

    return prompt

## 5. TODO: 프롬프트 설계
> 힌트: 모델에게 역할, 입력/출력 형식을 명확히 알려주세요.

In [9]:
def create_prompt(input_text, output_text=''):
    # TODO: 아래 messages 안의 "system" content 내용을 직접 설계해보세요.
    # 모델에게 줄 역할, 제약 조건, Few-shot(예시)을 자유롭게 작성하여 성능을 끌어올려 보세요!

    system_prompt = (
        # 역할 구체화
        "당신은 한국어 난독화 복원 전문가이다.\n"
        "난독화란 글자를 비슷한 발음, 비슷한 모양, 또는 자모 분리 등으로 변형한 것이다.\n"

        # 제약조건 엄격화
        "반드시 복원된 문장만 출력하라.\n"
        "원문의 띄어쓰기와 문장 구조를 최대한 유지하라.\n"
        "절대 설명, 번역, 부가설명을 추가하지 마라.\n"
        "Output 이후에 아무것도 출력하지 마라.\n"

        # 샘플 추가
        "Example 1:\n"
        "Input: 배쏭 개빠름 진짜 죠아용\n"
        "Output: 배송 개빠름 진짜 좋아용\n\n"

        "Example 2:\n"
        "Input: 마싯구 양 많아여!!\n"
        "Output: 맛있고 양 많아요!!\n\n"

        "Example 3:\n"
        "Input: 잚많 쟉꼬 갉 태 좋눼욥\n"
        "Output: 잠만 자고 갈 때 좋네요\n\n"

        "Example 4:\n"
        "Input: 절테 간면 않 된는 굣 멥몫\n"
        "Output: 절대 가면 안 되는 곳 메모\n\n"

        "Example 5:\n"
        "Input: 야... 칵컥 좋꾜 부됴 뼝 뚫렷썹\n"
        "Output: 아... 가격 좋고 뷰도 뻥 뚫려서\n\n"
    )

    prompt = (
        f"{system_prompt}"
        f"Input: {input_text}\n"
        f"Output: "
    )

    if output_text:
        prompt += output_text

    return prompt

## 6. 파인튜닝 전 추론 테스트

In [10]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
)

print('=== 파인튜닝 전 추론 결과 ===')
for _, row in test.head(3).iterrows():
    prompt = create_prompt(row['input'])
    output = pipe(prompt, max_new_tokens=50, do_sample=False, eos_token_id=tokenizer.eos_token_id)
    print(f'입력: {row["input"]}')
    print(f'출력: {output[0]["generated_text"][len(prompt):].strip()}')
    print()


[transformers] Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 파인튜닝 전 추론 결과 ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 녀뮨넒뭅 만죡숭러윤 효템뤼에오. 푸싸눼 옰면 콕 츄쩐학꼬 싶은 콧쉰웨오. 췌꾜윕뉘댜! ㅎㅎ 당음웨 또 옭 컷 갗았요.
출력: 녀석 멀리서 봐도 앜 앜 앜 앜 앜 앜 앜 앜 앜 앜



[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 풀룐투갸 엎코, 좀식또 업읍머, 윌뱐 잎츔민든릿 샤있샤윔엡 위썬 호뗄첨렴 관뤽갉 찰 앉 뙨는 누뀜뮈넬오. 까썽뷔갚 떨여쳐옵.
출력: 풀어 엎어, 좀식 잎새 땔 벗어, 윌 잎새 땔 벗어, 잎새 땔 벗어, 잎새 ��

입력: 쥔차 붉찐졀행욘. 삶먼섶 멂묽럿턴 혹텔 중웨 쬐약위였습뉜따. 칙어뉜쥐 샤쨩윈쥐 쩨끄윈할 땝붇텄 찐쩔함 1됴 없섣규욤. 3인 옌악학꼬 츄갚 쿰맥카찔 껄졔헷눈테 슉켠 츄까료 오청두링닒 쿤쩡썅 쑤컨 4께팎웨 첵콩인 않 된댜녜욥. 2쉬갼 졍돈 웨쭐핥꼬 닸싯 틀역칼 태 몄 효쉰냐, 2뿐 얘약햐쥐 않앝냥 묽엽봇썼어오(췌큼인핥 떼와 갇툰 뷴). 츄같큼 컬줴했교 깟트 껼젠 네억 뮨쟝됴 뽀없둘렸는뎁또 깟둡변홅 몃 뻔윈진 걺젤한 쉭갼, 푼, 촘꺄쥣 뮬엽봇쒸떪라규오. 걸쳬한 엉쑤중 칙쩝 짱즛쉰 닻음, 홋숫 찰못 져겼탸 한맏띠 핥셨엷오. 싻괏 한 먀딕 엾읊셧규오. 엎윅까 엾꼬 샹닿힐 풀쾨햇쑵뉘닯. 구릭코 냉쟝교엣써 섕선 퓔린냇 낢오. 묽엣써툐 핑륀맡 낮옆욤…
출력: 쥔차 붉은색 행복해요. 삶은 맑고 깨끗해요. 섕선 맑고 깨끗해요. 섕선 맑고 깨끗해요



## 7. 데이터셋 포맷 변환 (파인튜닝용)

In [11]:
def format_chat_template(row):
    prompt = create_prompt(row['input'], row['output'])
    tokens = tokenizer.encode(prompt, truncation=True, max_length=256)
    row['input_ids'] = tokens
    return row

# TODO: train_data, val_data를 Dataset으로 변환하세요
train_dataset = Dataset.from_pandas(train_data.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_data.reset_index(drop=True))

train_dataset = train_dataset.map(format_chat_template, batched=False)
val_dataset = val_dataset.map(format_chat_template, batched=False)


Map:   0%|          | 0/10136 [00:00<?, ? examples/s]

Map:   0%|          | 0/1127 [00:00<?, ? examples/s]

## 8. LoRA 파인튜닝

Q (Query) - q_proj
"내가 뭘 찾고 있지?" 역할
현재 단어가 다른 단어들한테 질문하는 것

K (Key) - k_proj
"나는 이런 정보를 갖고 있어" 역할
각 단어가 자신을 소개하는 것

V (Value) - v_proj
"실제 내용은 이거야" 역할
K가 선택되면 실제로 가져오는 정보

Out (Output) - out_proj
Q, K, V 계산 결과를 최종 출력으로 변환하는 역할

In [26]:
!pip install -U trl peft transformers bitsandbytes accelerate

In [29]:
import trl
print(trl.__version__)

1.6.0


In [39]:
from trl import SFTTrainer
from transformers import TrainingArguments
from peft import LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],  # 힌트: 성능을 더 높이고 싶다면 다른 module을 추가해보세요! => 넹
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.train()

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    # Colab 무료(T4 GPU) 환경을 위해 fp16을 True로 설정합니다.
    fp16=True,
    bf16=False,

    logging_steps=10,

    eval_strategy='steps',
    eval_steps=50,
    save_strategy='epoch',
    report_to='none'
)


trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=128,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

ADAPTER_MODEL = 'lora_adapter_2b'
trainer.model.save_pretrained(ADAPTER_MODEL)


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'

In [2]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.train()

training_args = SFTConfig(
    output_dir='./results',
    num_train_epochs=1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    fp16=False,
    bf16=False,

    optim='paged_adamw_8bit',

    logging_steps=20,
    eval_strategy='steps',
    eval_steps=150,
    save_strategy='epoch',

    report_to='none',
    dataset_text_field='text',
    max_length=128,
)

print("fp16:", training_args.fp16)
print("bf16:", training_args.bf16)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset.shuffle(seed=42),
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
    #최신 버전에는 안 씀 dataset_text_field="text",
    #최신 버전에는 안 씀 max_seq_length=128,
)

trainer.train()

ADAPTER_MODEL = 'lora_adapter_2b'
trainer.model.save_pretrained(ADAPTER_MODEL)
tokenizer.save_pretrained(ADAPTER_MODEL)

ModuleNotFoundError: No module named 'trl'

In [3]:
import os

os.listdir('lora_adapter_2b')

FileNotFoundError: [Errno 2] No such file or directory: 'lora_adapter_2b'

## 9. 파인튜닝된 모델로 추론

In [ ]:
!pip install -U torchao

In [ ]:
BASE_MODEL = 'beomi/gemma-ko-2b'
ADAPTER_MODEL = 'lora_adapter_2b'

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map='auto',
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_MODEL
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer
)

In [ ]:
!zip -r lora_adapter_2b.zip lora_adapter_2b

## 10. TODO: Validation Accuracy 측정
> 힌트: `val_data`의 `output` 컬럼과 모델 예측 결과를 비교하세요.

In [ ]:
from collections import Counter

def char_f1(pred, answer):
    pred_chars = Counter(pred)
    ans_chars = Counter(answer)

   # ----------------------------------------------------
   # TODO: 두 문장의 공통 문자 개수를 계산하세요.
   # 힌트: Counter 객체의 교집합(&) 연산과 .values()를 활용해보세요.
   # ----------------------------------------------------
    common = sum((pred_chars & ans_chars).values())

    if common == 0:
        return 0.0

    # Precision, Recall, F1 계산
    precision = common / sum(pred_chars.values())
    recall = common / sum(ans_chars.values())
    f1 = 2 * (precision * recall) / (precision + recall)

    return f1

    # ----------------------------------------------------
    # TODO: Precision, Recall 공식을 이용해 F1 Score를 구하세요.
    # 공식: F1 = 2 * (Precision * Recall) / (Precision + Recall)
    # ----------------------------------------------------
        # Precision, Recall, F1 계산
    precision = common / sum(pred_chars.values())
    recall = common / sum(ans_chars.values())
    f1 = 2 * (precision * recall) / (precision + recall)

    return f1

val_preds = []
val_sample = val_data.head(30)

for _, row in val_sample.iterrows():
    prompt = create_prompt(row['input'])
    output = pipe(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )
    # LLM 특성상 입력한 프롬프트 뒤에 정답을 이어서 생성하므로, 프롬프트 길이만큼 슬라이싱합니다.
    val_preds.append(
        output[0]['generated_text'][len(prompt):].strip()
    )

# Exact Match 대신 문자 단위 F1
scores = [char_f1(pred, ans) for pred, ans in zip(val_preds, val_sample['output'])]
print(f'Validation Char F1: {sum(scores)/len(scores):.4f}')



from collections import Counter

def char_f1(pred, answer):
    pred_chars = Counter(pred)
    ans_chars = Counter(answer)

    # 두 문장의 공통 문자 개수 계산
    common = sum((pred_chars & ans_chars).values())

    if common == 0:
        return 0.0

    # Precision, Recall, F1 계산
    precision = common / sum(pred_chars.values())
    recall = common / sum(ans_chars.values())
    f1 = 2 * (precision * recall) / (precision + recall)

    return f1

val_preds = []
val_sample = val_data.head(30)

for _, row in val_sample.iterrows():
    prompt = create_prompt(row['input'])
    output = pipe(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )
    val_preds.append(
        output[0]['generated_text'][len(prompt):].strip()
    )

scores = [char_f1(pred, ans) for pred, ans in zip(val_preds, val_sample['output'])]
print(f'Validation Char F1: {sum(scores)/len(scores):.4f}')

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Validation Char F1: 0.1409


In [ ]:
restored_reviews = []

test_prompts = [create_prompt(text) for text in test['input'].tolist()]
outputs = pipe(
    test_prompts,
    max_new_tokens=50,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    batch_size=4  # 메모리 상황에 따라 4~16 조절 가능
)

for prompt, output in zip(test_prompts, outputs):
    generated_text = output['generated_text']
    result = generated_text[len(prompt):].strip()
    restored_reviews.append(result)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

## 11. Todo: 제출 파일 생성

submission_gemma.csv 저장 완료!
